# Overfitting & Validation : Notebook de l'après-midi
## Épisodes 5 à 8

Suite du notebook de la matinée. Mêmes règles de lecture : **exécutée** / **à trous** (les `___` portent le principe) / **ce que vous devez voir** / **auto-explication** / **[L3+]** / **[M2]**. Graphiques en erreur, notes /20 dans les tables.

Rappels d'une ligne : un élève = un modèle, **aucune initiative** ; le prof = le data scientist, c'est lui qui compare des configurations et qui se souvient des notes ; `verite_du_magicien()` = la vraie erreur, cellules d'affichage seulement ; l'examen final est **scellé** : plus pour longtemps.

**Le fil de l'après-midi.**

| Épisode | Question de l'histoire | Ce qu'on apprend |
|---|---|---|
| **5** | Le prof garde la meilleure note des vendredis : peut-il l'annoncer ? | Sélectionner consomme la note ; CV imbriquée ; **la levée du sceau** |
| **6** | Le magicien rate son tour, exprès | Les fuites ; le pipeline comme contrat |
| **7** | La fiche d'une page | Régularisation : Ridge et Lasso |
| **8** | Trois élèves mystère ; et l'élève qui mémorise tout | Le diagnostic ; où l'histoire casse (2/2) |

**Exécutée** : on recharge le décor, qui s'agrandit, deux nouveaux livres apparaissent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

SIGMA = 0.3          # amplitude du bruit : les fautes de frappe dans les corrigés
N_LIVRE = 30         # exercices par livre
N_VENDREDIS = 5      # 5 vendredis x 6 exercices


def concepts(x):
    """Les concepts de physique. Réservé au mode omniscient du magicien."""
    return np.sin(2 * np.pi * x)


def fonds_exercices(n, seed):
    """Tire n exercices du fonds du prof : énoncé x, corrigé y (avec son bruit)."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, size=n)
    y = concepts(x) + rng.normal(0, SIGMA, size=n)
    return x[:, None], y


def magicien(X, y, vendredi):
    """Pouvoir 1, le tour : retire les exercices du vendredi demandé (1..N_VENDREDIS).
    Trois lignes. N'importe quel prof peut le faire."""
    plis = list(KFold(N_VENDREDIS, shuffle=True, random_state=0).split(X))
    idx_visible, idx_disparus = plis[vendredi - 1]
    return (X[idx_visible], y[idx_visible]), (X[idx_disparus], y[idx_disparus])


_X_INFINI, _Y_INFINI = fonds_exercices(100_000, seed=424242)   # « une infinité de livres »


def verite_du_magicien(eleve):
    """Pouvoir 2, le mode omniscient : la note de l'élève face à la vérité absolue
    du magicien (10^5 exercices frais). À ne pas confondre avec l'examen final,
    qui est un livre fini que l'élève passera vraiment.
    Cellules d'affichage uniquement. Ne sert JAMAIS à choisir quoi que ce soit."""
    return mean_squared_error(_Y_INFINI, eleve.predict(_X_INFINI))


BAREME = 0.5 + SIGMA**2   # variance des corrigés du fonds : réciter la réponse moyenne partout -> 0/20


def en_note(erreur):
    """Le barème du prof : convertit l'erreur de la machine en note sur 20.
    20/20 = zéro erreur ; 0/20 = réciter la même réponse à tous les exercices.
    Toujours : moins d'erreur = meilleure note. Le barème ne change jamais.
    Accepte un nombre ou un tableau."""
    return np.maximum(0.0, 20 * (1 - np.asarray(erreur) / BAREME))


sceau_leve = False


def examen_final():
    """L'autre livre, mêmes concepts. Scellé jusqu'à l'épisode 5."""
    if not sceau_leve:
        raise RuntimeError("L'examen final est scellé jusqu'à l'épisode 5.")
    return fonds_exercices(N_LIVRE, seed=2025)


def eleve(nb_regles):
    """Un élève défini par le nombre de règles qu'il retient de ses révisions :
    1 règle = l'élève naïf ; quelques-unes = le généraliste ; beaucoup = le par-cœur.
    (Sous le capot : un polynôme, la révélation est en fin de notebook.
    Recentrage des énoncés sur [-1, 1] puis mise à l'échelle des puissances : sans effet
    sur la courbe, seulement sur la stabilité du calcul : le vrai par-cœur, sur toutes les machines.)"""
    return make_pipeline(MinMaxScaler(feature_range=(-1, 1)),
                         PolynomialFeatures(nb_regles, include_bias=False),
                         StandardScaler(),
                         LinearRegression())


X_livre, y_livre = fonds_exercices(N_LIVRE, seed=16)                      # le livre est imprimé...
(X_vis, y_vis), (X_disp, y_disp) = magicien(X_livre, y_livre, vendredi=1)  # ...le tour est fait AVANT

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ---- Le décor s'agrandit : deux nouveaux livres pour l'après-midi ----

def fonds_chapitres(n, seed, rho=0.7):
    """Le livre à chapitres : chaque exercice est décrit par 50 indices (un par chapitre).
    Seuls 5 chapitres comptent vraiment : le magicien sait lesquels (beta_vrai)."""
    rng = np.random.default_rng(seed)
    idx = np.arange(50)
    L = np.linalg.cholesky(rho ** np.abs(idx[:, None] - idx[None, :]))   # chapitres voisins corrélés
    X = rng.normal(size=(n, 50)) @ L.T
    beta_vrai = np.zeros(50); beta_vrai[[0, 10, 20, 30, 40]] = [3, -2, 1.5, 2.5, -1]
    y = X @ beta_vrai + rng.normal(0, 1.0, size=n)                        # bruit sigma = 1
    return X, y, beta_vrai

X_chap, y_chap, beta_vrai = fonds_chapitres(100, seed=7)
_XC, _YC, _ = fonds_chapitres(50_000, seed=999)


def verite_du_magicien_chapitres(eleve):
    """Mode omniscient, version livre à chapitres (50 000 exercices frais). Plancher : sigma^2 = 1."""
    return mean_squared_error(_YC, eleve.predict(_XC))


def fonds_sans_concepts(n=40, p=1000, seed=3):
    """Le livre sans concepts : 1000 chapitres, et des corrigés tirés au hasard.
    Il n'y a RIEN à apprendre dedans : le magicien le sait."""
    rng = np.random.default_rng(seed)
    return rng.normal(size=(n, p)), rng.normal(size=n)

X_rien, y_rien = fonds_sans_concepts()

print(f"Décor rechargé. Livre du matin : {len(X_livre)} exercices. "
      f"Livre à chapitres : {X_chap.shape[0]} exercices x {X_chap.shape[1]} chapitres. "
      f"Livre sans concepts : {X_rien.shape[0]} x {X_rien.shape[1]}. Sceau : {'levé' if sceau_leve else 'intact'}.")

---
## Épisode 5 : Le prof réglé sur les vendredis

Le prof a pris goût au jeu de l'épisode 4. Toute l'année, il compare des configurations d'élèves (1 règle, 2 règles, … 10 règles) et garde celle du meilleur score aux vendredis. Fin juin, il rédige son rapport : *« Mon champion obtient la moyenne X aux vendredis ; c'est donc sa note attendue à l'examen. »*

Et c'est là qu'il se ment. Il a choisi le champion **parce que** les vendredis l'avaient récompensé. Sur dix moyennes bruitées, la plus haute est haute pour deux raisons mélangées : le talent du champion, *et* la chance qui l'a fait gagner. En annonçant la note qui a servi au choix, il annonce le talent **plus** la chance. Souvenez-vous de qui se souvient : les élèves ne se transmettent rien (épisode 4) ; c'est le prof qui a lu toutes les notes, et c'est sa mémoire à lui qui contamine le chiffre.

**Exécutée** : le magicien vérifie sur 40 années scolaires (40 livres frais). Pour chaque année : la note annoncée par le prof (`best_score_`), la note obtenue par une procédure honnête (sélection *refaite à l'intérieur* de chaque monde ; la CV imbriquée, cellule à trous juste après), et la vérité du champion.

In [ ]:
from sklearn.model_selection import GridSearchCV, ShuffleSplit, cross_val_score

grille_configs = {"polynomialfeatures__degree": np.arange(1, 11)}   # les 10 configurations du prof
N_ANNEES = 40

annoncee, honnete, verite_champion = [], [], []
for i in range(N_ANNEES):
    X, y = fonds_exercices(N_LIVRE, seed=9000 + i)                  # une année scolaire = un livre frais
    interieur = KFold(5, shuffle=True, random_state=i)

    prof = GridSearchCV(eleve(1), grille_configs, cv=interieur,
                        scoring="neg_mean_squared_error").fit(X, y)
    annoncee.append(-prof.best_score_)                              # la note que le prof annonce
    verite_champion.append(verite_du_magicien(prof.best_estimator_))

    honnete.append(-cross_val_score(                                # sélection refaite DANS chaque monde
        GridSearchCV(eleve(1), grille_configs, cv=interieur, scoring="neg_mean_squared_error"),
        X, y, cv=KFold(5, shuffle=True, random_state=1000 + i),
        scoring="neg_mean_squared_error").mean())

plt.figure(figsize=(7.5, 4.2))
plt.boxplot([annoncee, honnete, verite_champion],
            tick_labels=["note annoncée\n(best_score_)", "procédure honnête\n(CV imbriquée)", "vérité du magicien\n(champion de l'année)"])
plt.ylabel("erreur (plus bas = meilleure note)"); plt.title(f"{N_ANNEES} années scolaires, trois façons de chiffrer le même champion")
plt.show()
print(f"moyennes. annoncée : {np.mean(annoncee):.3f} | honnête : {np.mean(honnete):.3f} | vérité : {np.mean(verite_champion):.3f}")

**Ce que vous devez voir.** Trois boîtes pour le même champion : la note annoncée (≈ 0.10) est systématiquement **plus belle** que ce que le champion vaut vraiment (≈ 0.15). L'écart n'est pas une malchance, c'est un mécanisme : le maximum de dix moyennes bruitées est en moyenne au-dessus du talent du meilleur ; la **malédiction du vainqueur**. La procédure honnête (≈ 0.13) recolle à la réalité : elle refait la sélection *à l'intérieur* de chaque monde, de sorte qu'aucune note utilisée pour choisir ne soit jamais annoncée.

**À trous** : l'idiome de la procédure honnête, à savoir écrire les yeux fermés, une sélection (boucle intérieure) **emboîtée** dans une évaluation (boucle extérieure). En scikit-learn, ça tient en une ligne.

In [ ]:
selection = GridSearchCV(eleve(1), grille_configs, cv=5, scoring="neg_mean_squared_error")

note_honnete = -cross_val_score(___,                      # À COMPLÉTER : QUI évalue-t-on ? (la sélection entière, pas un élève)
                                X_livre, y_livre,
                                cv=___,                   # À COMPLÉTER : la boucle EXTÉRIEURE (5 mondes)
                                scoring="neg_mean_squared_error").mean()
print(f"Sur le livre du matin, note honnête de la procédure : {note_honnete:.3f} soit {en_note(note_honnete):.1f}/20")
print("(Sur UN livre, honnête et annoncée peuvent coïncider ; le biais est une moyenne : cf. les 40 années.)")

**[M2] Encadré : ce que dit la littérature.** Varma & Simon (*BMC Bioinformatics*, 2006) ont mesuré exactement ce phénomène sur des jeux *sans aucun signal* : la CV utilisée à la fois pour régler et pour annoncer donne des erreurs très optimistes, alors que la CV imbriquée est presque sans biais. Le mécanisme est le vôtre : minimum de M estimations bruitées ⇒ E[min] < min E, et l'écart grandit avec M. Arlot & Celisse (2010) posent la distinction générale : la CV sert soit à **estimer**, soit à **sélectionner** : jamais les deux avec le même chiffre. *On revient au prof.*

**La levée du sceau.** Il reste une troisième voie, la plus simple de toutes : garder un livre que **personne** n'a jamais ouvert, ni élève, ni prof, et le sortir une seule fois, tout à la fin. Ce livre existe depuis l'épisode 0. Le moment est venu.

**Exécutée** : on lève le sceau. Une fois.

In [ ]:
champion = GridSearchCV(eleve(1), grille_configs, cv=KFold(5, shuffle=True, random_state=0),
                        scoring="neg_mean_squared_error").fit(X_livre, y_livre)
print(f"Champion de l'année : {champion.best_params_['polynomialfeatures__degree']} règles.")
print(f"Note annoncée par le prof (best_score_) : {en_note(-champion.best_score_):.1f}/20\n")

sceau_leve = True                                          # ce geste ne se fait qu'UNE fois
X_examen, y_examen = examen_final()
erreur_examen = mean_squared_error(y_examen, champion.predict(X_examen))

print(f"EXAMEN FINAL ({len(X_examen)} exercices jamais vus) : {erreur_examen:.3f} soit {en_note(erreur_examen):.1f}/20")
print(f"Vérité du magicien pour ce champion          : {verite_du_magicien(champion.best_estimator_):.3f} soit {en_note(verite_du_magicien(champion.best_estimator_)):.1f}/20")
print("\nL'examen ne tombe pas pile sur la vérité : 30 exercices, c'est une note, pas la vérité.")
print("On ne le repasse pas. Si on recommençait « jusqu'à ce que ça tombe bien », il deviendrait un vendredi de plus.")

**Auto-explication** : *Le prof annonce la meilleure moyenne des vendredis comme note d'examen attendue. En une phrase de l'histoire : pourquoi ce chiffre est-il trop beau ?*

> Votre réponse :

**[L3+] Extension** : l'optimisme grandit avec le nombre de candidats. Reprenez les 40 années de la première cellule et comparez : si le prof n'avait comparé que 3 configurations (1 à 3 règles), l'écart entre note annoncée et vérité aurait-il été plus petit qu'avec 10 ? Mesurez les deux.

In [ ]:
# Votre code ici


---
## Épisode 6 : Le tour raté

Aujourd'hui, le magicien rate son tour. **Exprès**, pour montrer ce qui se passe. La cause est unique : les exercices disparaissent **trop tard** : après que quelque chose a déjà été fait avec le livre entier. Trois cas :

1. L'élève a calibré sa fiche (unités, ordres de grandeur) sur tout le livre, *puis* le magicien retire les exercices. : Standardiser sur tout, puis découper.
2. L'élève a choisi ses chapitres à réviser en regardant tout le livre, *puis* le magicien retire les exercices. : Sélectionner les variables sur tout, puis découper.
3. Le magicien retire un exercice mais en laisse un double ailleurs dans le livre. : Doublons.

Le critère n'est **pas** « a-t-on regardé les corrigés du vendredi ». C'est : **la procédure ajustée a-t-elle vu, d'une façon ou d'une autre, les exercices du vendredi ?** Pour le mesurer sans triche possible, on prend le pire terrain : le **livre sans concepts** : 1000 chapitres, des corrigés tirés au hasard, rien à apprendre. Sur ce livre, toute note supérieure à « je récite la moyenne » est un mensonge.

**Exécutée** : le cas 2, version ratée, on choisit les 10 chapitres « les plus prometteurs » en regardant le livre **entier**, puis seulement on fait tourner les vendredis.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

choix_trop_tot = SelectKBest(f_regression, k=10).fit(X_rien, y_rien)      # a vu TOUT le livre...
X_choisis = choix_trop_tot.transform(X_rien)

r2_fuite = cross_val_score(LinearRegression(), X_choisis, y_rien, cv=5, scoring="r2").mean()
print(f"Livre SANS CONCEPTS, chapitres choisis avant le tour : R² des vendredis = {r2_fuite:+.2f}")
print("R² > 0 : les vendredis jurent qu'on prédit... un livre où il n'y a rien à prédire.")

**À trous** : le tour réussi, le choix des chapitres doit se faire **dans chaque monde**, sur le livre visible de ce monde seulement. En scikit-learn, ça s'écrit en mettant le sélecteur *dans* le pipeline : à chaque pli, `fit` (sélection + élève) ne voit que le pli d'entraînement.

In [ ]:
tour_reussi = Pipeline([
    ("choix_chapitres", ___),        # À COMPLÉTER : le même sélecteur (10 chapitres)... mais DANS le pipeline
    ("eleve", ___),                  # À COMPLÉTER : le même élève
])
r2_honnete = cross_val_score(tour_reussi, X_rien, y_rien, cv=5, scoring="r2").mean()
print(f"Même sélecteur, même élève, même k, dans le pipeline : R² = {r2_honnete:+.2f}")

**Ce que vous devez voir.** Le même sélecteur, le même élève, le même découpage : **+0.48 hors pipeline, négatif dedans.** Rien d'autre n'a changé que le *moment* où le choix des chapitres est fait. Une demi-unité de R² fabriquée à partir de bruit pur, voilà ce que coûte un tour raté, et voilà pourquoi il se rate si facilement : le code fautif a l'air parfaitement raisonnable.

**[L3+] Encadré : le pipeline comme contrat.** À chaque pli, scikit-learn appelle `fit(transformations + élève)` sur le pli d'entraînement **seulement**, puis `transform` + `predict` sur le pli de validation. La règle tient en une phrase : **tout ce qui apprend quelque chose des données va dans le pipeline** : sélection de chapitres, standardisation, imputation, réduction de dimension. La doc scikit-learn (*Common pitfalls*) utilise exactement notre cas et précise que le risque vaut pour presque toutes les transformations. *On revient au magicien.*

**Auto-explication** : *Dans le cas 2, l'élève n'a jamais lu les corrigés des exercices disparus ; seulement leurs énoncés, pour choisir ses chapitres. Est-ce une fuite ?*

> Votre réponse :

**[L3+] Extension** : le cas 1 (la fiche calibrée sur tout le livre), sur le **livre à chapitres**, standardisez avant le découpage puis dans le pipeline, et mesurez l'écart entre les deux erreurs de CV. Que concluez-vous sur le rapport entre le *principe* et l'*ampleur* ?

In [ ]:
# Votre code ici


**[M2] Plafond : le cas 3 et la taxonomie.** Kapoor & Narayanan (*Patterns*, 2023) recensent au moins 294 articles touchés par des fuites dans 17 disciplines et en proposent une taxonomie de 8 types ; nos trois cas s'y rangent (prétraitement sur train + test, sélection sur train + test, doublons). Implémentez le cas 3 sur le livre du matin : glissez des copies des exercices de l'examen parmi les exercices de révision d'un élève par cœur, et mesurez la note gonflée.

In [ ]:
# Votre code ici


---
## Épisode 7 : La fiche d'une page

Le prof en a assez du par-cœur. Plutôt que de limiter le *nombre* de règles (le levier de l'épisode 2), il impose une contrainte nouvelle : **une fiche d'une page**. L'élève garde ses 50 chapitres sous les yeux, mais la place est comptée : impossible d'écrire 50 règles longues. Il doit compresser. Deux stratégies de fiche :

- **Raccourcir tous les arguments** : garder les 50 règles, mais toutes plus courtes. C'est **Ridge**.
- **N'en garder que quelques-unes** : écrire en grand les règles qui comptent, rayer les autres. C'est **Lasso**.

Le terrain : le **livre à chapitres** : 100 exercices, 50 chapitres, et le magicien sait que 5 chapitres seulement comptent (`beta_vrai`). La longueur de la page (λ) ? Elle n'est pas devinée : elle est **choisie sur les vendredis** (épisode 4), par une sélection honnête (épisode 5), avec la fiche écrite sur le livre visible seulement (épisode 6). Tout l'après-midi converge ici.

**Exécutée** : le chemin des fiches, ce que deviennent les 50 règles quand la page rétrécit (λ augmente, de droite à gauche... non : regardez l'axe).

In [ ]:
from sklearn.linear_model import lasso_path, Ridge

X_std = StandardScaler().fit_transform(X_chap)             # pour VISUALISER les chemins seulement
alphas = np.logspace(-3, 3, 60)

alphas_l, coefs_l, _ = lasso_path(X_std, y_chap, alphas=alphas)
coefs_r = np.array([Ridge(alpha=a).fit(X_std, y_chap).coef_ for a in alphas]).T

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)
for ax, coefs, titre in [(axes[0], coefs_r, "Ridge : raccourcir toutes les règles"),
                         (axes[1], coefs_l, "Lasso : n'en garder que quelques-unes")]:
    for j in range(50):
        ax.plot(alphas if titre.startswith("Ridge") else alphas_l, coefs[j],
                color=("C3" if beta_vrai[j] != 0 else "gray"),
                lw=(2 if beta_vrai[j] != 0 else 0.6), alpha=(1 if beta_vrai[j] != 0 else 0.5))
    ax.set_xscale("log"); ax.set_xlabel("λ (page de plus en plus courte →)"); ax.set_title(titre)
axes[0].set_ylabel("taille de chaque règle (coefficient)")
plt.suptitle("50 règles face à la fiche d'une page : en rouge, les 5 chapitres qui comptent vraiment (secret du magicien)")
plt.tight_layout(); plt.show()

**Ce que vous devez voir.** À gauche (Ridge), les 50 règles **rétrécissent ensemble**, continûment ; aucune ne devient exactement nulle. À droite (Lasso), les règles **s'éteignent une à une** quand la page raccourcit, et les cinq rouges (les vrais chapitres, que seul le magicien connaît) sont parmi les dernières à survivre. Rayer, c'est sélectionner.

**À trous** : écrire les deux fiches pour de vrai. Tout l'après-midi tient dans cette cellule : la fiche est un pipeline (épisode 6), et λ est choisi par validation croisée (épisodes 4–5).

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

fiche_ridge = make_pipeline(StandardScaler(), ___(alphas=np.logspace(-3, 3, 60), cv=5))   # À COMPLÉTER : quelle fiche raccourcit tout ?
fiche_lasso = make_pipeline(StandardScaler(), ___(cv=5, random_state=0))                  # À COMPLÉTER : quelle fiche raye ?
sans_fiche  = make_pipeline(StandardScaler(), LinearRegression())

for nom, m in [("sans fiche (50 règles libres)", sans_fiche), ("fiche Ridge", fiche_ridge), ("fiche Lasso", fiche_lasso)]:
    m.fit(X_chap, y_chap)
    print(f"{nom:30s} vérité du magicien : {verite_du_magicien_chapitres(m):.3f}   (plancher bruit σ² = 1)")

**Exécutée** : la fiche Lasso, chapitre par chapitre, face au secret du magicien.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
largeur = 0.4
ax.bar(np.arange(50) - largeur/2, beta_vrai, largeur, label="secret du magicien (β vrai)", color="C0")
ax.bar(np.arange(50) + largeur/2, fiche_lasso[-1].coef_ / fiche_lasso[0].scale_, largeur,
       label="fiche Lasso (règles, remises à l'échelle d'origine)", color="C3")
ax.set_xlabel("chapitre"); ax.set_ylabel("poids de la règle"); ax.legend(fontsize=8)
n_gardees = int(np.sum(np.abs(fiche_lasso[-1].coef_) > 1e-6))
ax.set_title(f"La fiche Lasso garde {n_gardees} règles sur 50 : les 5 vraies y sont")
plt.tight_layout(); plt.show()

**Ce que vous devez voir.** Sans fiche, 50 règles libres : vérité ≈ 1.71. Fiche Ridge : ≈ 1.67, un gain modeste, toutes les règles survivent en plus court. Fiche Lasso : ≈ 1.31, nettement mieux, une douzaine de règles gardées dont les 5 vraies. Personne sous 1.0 : c'est le bruit. La fiche ne rend pas l'élève plus savant : elle **l'empêche de gaspiller sa page sur du bruit** : c'est le levier variance de l'épisode 3, actionné sans changer d'élève.

**Auto-explication** : *La fiche « n'en garder que quelques-unes » retrouve les 5 bons chapitres ; la fiche « tout en plus court » garde 50 barres et gagne à peine. Dans les mots de l'histoire, pourquoi ?*

> Votre réponse :

**[L3+] Encadré : les deux pénalités.** min ‖y − Xβ‖² + λ‖β‖₂² (Ridge) · min ‖y − Xβ‖² + λ‖β‖₁ (Lasso). λ ↑ = page plus courte. λ est un hyperparamètre : choisi par CV, jamais sur l'examen (E5) ; et X est standardisé **dans le pipeline** (E6), sinon la pénalité taxe les chapitres selon leurs unités. *On revient à la fiche.*

**[L3+] Extension** : entre les deux, `ElasticNetCV` mélange les pénalités (`l1_ratio`). Balayez `l1_ratio` ∈ {0.1, 0.5, 0.9, 1.0} et regardez vérité et nombre de règles gardées.

In [ ]:
# Votre code ici


**[M2] Plafond : les chapitres photocopiés.** Fabriquez un petit livre où deux chapitres sont quasi identiques (corrélation ≈ 0.999) et comptent autant l'un que l'autre. Ajustez la fiche Lasso sur dix tirages du livre, puis une fiche Ridge à pénalité fixée. Qu'observez-vous sur les deux premières règles, et qu'en concluez-vous pour l'*interprétation* d'une fiche Lasso sous forte corrélation ?

**[M2] Encadré : la géométrie, en une image.** Contrainte ‖β‖₁ ≤ t : un losange, dont les coins sont sur les axes, l'optimum tombe souvent sur un coin, d'où les zéros exacts. Contrainte ‖β‖₂² ≤ t : un disque, sans coin ; jamais de zéro exact. (ISLR, ch. 6.)

In [ ]:
# Votre code ici


---
## Épisode 8 : Le défi diagnostic, et là où l'histoire casse

**Le défi `[Tous]`.** Trois élèves mystère, A, B, C, dont le prof a caché la configuration (nombre de règles, nombre d'exercices travaillés). Vous n'avez droit qu'à leurs **courbes** : la courbe d'apprentissage de leur configuration (erreur livre / erreur vendredis en fonction du nombre d'exercices), avec un point marquant où l'élève se trouve. Pour chacun, répondez : **quel levier tourner** : plus d'exercices (tome 2), moins de règles (fiche), ou ne rien changer ? C'est l'intitulé officiel de ce lab, *diagnostic de modèles sur-entraînés*.

**Exécutée** : les trois bulletins. (La cellule qui suit contient les configurations, encodées pour ne pas vendre la réponse : ne la décodez pas avant d'avoir répondu.)

In [ ]:
import base64, json
_secret = json.loads(base64.b64decode(b'eyJBIjogWzEsIDIwMF0sICJCIjogWzEyLCAyNV0sICJDIjogWzMsIDM1XX0=').decode())

def bulletin(nom):
    """Trace la courbe d'apprentissage de la configuration cachée de l'élève, et marque où il est."""
    r, n_actuel = _secret[nom]
    X_t, y_t = fonds_exercices(250, seed=2)
    tailles = np.array(sorted(set([16, 24, 32, 48, 64, 96, 128, 160, 200] + [n_actuel])))
    n_ex, tr, va = learning_curve(eleve(r), X_t, y_t, train_sizes=tailles,
                                  cv=KFold(5, shuffle=True, random_state=0),
                                  scoring="neg_mean_squared_error", shuffle=True, random_state=0)
    plt.figure(figsize=(6.2, 3.6))
    plt.plot(n_ex, -tr.mean(axis=1), "o-", label="erreur sur le livre")
    plt.plot(n_ex, -va.mean(axis=1), "s-", label="erreur des vendredis")
    i = int(np.where(n_ex == n_actuel)[0][0])
    plt.scatter([n_actuel], [-va.mean(axis=1)[i]], s=180, facecolors="none", edgecolors="k",
                lw=2, zorder=5, label="l'élève est ICI")
    plt.axhline(SIGMA**2, color="gray", ls=":", label="bruit σ²")
    plt.yscale("log"); plt.xlabel("nombre d'exercices travaillés"); plt.ylabel("erreur")
    plt.title(f"Bulletin de l'élève {nom}"); plt.legend(fontsize=8); plt.show()

from sklearn.model_selection import learning_curve
for nom_eleve in ["A", "B", "C"]:
    bulletin(nom_eleve)

**Votre diagnostic** (trois lignes, une par élève : quel levier, et pourquoi) :

> A :
> B :
> C :

---
### `[M2]` Où l'histoire casse (2/2) : l'élève qui mémorise tout... sobrement

Un dernier personnage, pour les M2. Un élève reçoit de plus en plus de chapitres à disposition : bientôt plus de chapitres que d'exercices. Passé ce point, il peut **tout mémoriser** : zéro erreur sur le livre, toujours. L'histoire de l'épisode 1 prédit la catastrophe. Et elle a raison... jusqu'au seuil. Car parmi *toutes* les façons de tout mémoriser, cet élève-là choisit systématiquement **la plus sobre** : celle dont les règles sont les plus courtes possibles (c'est ce que fait `LinearRegression`, via `lstsq`, quand p > n, la solution de norme minimale). Et sa vérité **redescend**.

L'histoire n'a pas de mot pour ça : « mémoriser » y est une chose, pas une famille de solutions parmi lesquelles on peut être sobre. C'est le deuxième et dernier endroit où le décor casse, et il ne casse que dans ce régime précis, en régression linéaire de norme minimale (Belkin, Hsu, Ma & Mandal, *PNAS* 2019). Aucune autre famille de modèles n'est en jeu ici.

**Exécutée** : n = 40 exercices fixes ; on donne à l'élève de 1 à 80 chapitres.

In [ ]:
rng = np.random.default_rng(0)
n, p_total = 40, 80
beta80 = rng.normal(size=p_total) / np.sqrt(p_total)                     # les concepts vivent dans 80 chapitres
X80 = rng.normal(size=(n, p_total)); y80 = X80 @ beta80 + rng.normal(0, 0.5, size=n)
X80_frais = rng.normal(size=(20_000, p_total)); y80_frais = X80_frais @ beta80 + rng.normal(0, 0.5, size=20_000)

nb_chapitres = np.arange(1, p_total + 1)
verites_dd = [mean_squared_error(y80_frais, LinearRegression().fit(X80[:, :j], y80).predict(X80_frais[:, :j]))
              for j in nb_chapitres]

plt.figure(figsize=(7.5, 4.2))
plt.plot(nb_chapitres, verites_dd, "o-", ms=3)
plt.axvline(n, color="k", ls="--", label=f"seuil : autant de chapitres que d'exercices (p = n = {n})")
plt.axhline(0.25, color="gray", ls=":", label="bruit σ² = 0.25")
plt.yscale("log"); plt.xlabel("chapitres à disposition (p)"); plt.ylabel("vérité (erreur sur 20 000 exercices frais)")
plt.title("La double descente : régression linéaire de norme minimale"); plt.legend(fontsize=8); plt.show()
print(f"vérité à p=36 : {verites_dd[35]:.2f} | au seuil p=40 : {verites_dd[39]:.0f} | à p=80 : {verites_dd[79]:.2f}")

---
## La charte, ce qu'on emporte

Cinq règles, chacune dans l'histoire puis en clair :

1. **L'examen final se passe une fois.** Le jeu de test est touché une seule fois, à la fin : le sceau ne se relève pas.
2. **Toute note lue pour choisir est consommée.** Sélectionner, c'est entraîner (le prof aussi) ; validation ≠ test ; la sélection se refait *dans* la boucle (CV imbriquée).
3. **Deux leviers, et un diagnostic d'abord.** Le tome 2 (n) ou la fiche (capacité, λ) : la courbe d'apprentissage dit lequel tourner, et parfois de ne rien toucher.
4. **Le tour du magicien se fait en premier, et à chaque monde.** Tout ce qui apprend des données va dans le pipeline.
5. **Une note est une estimation.** La reporter avec sa dispersion ; l'examen final lui-même est une note.

---
## Lexique FR ↔ EN (l'après-midi)

| Dans l'histoire | En français technique | En anglais (doc, API) |
|---|---|---|
| La note consommée par le choix | Biais de sélection, malédiction du vainqueur | Selection bias, winner's curse |
| La sélection refaite dans chaque monde | Validation croisée imbriquée | Nested cross-validation |
| La note annoncée par le prof | Score de la meilleure configuration | `GridSearchCV.best_score_` |
| Lever le sceau | Évaluer sur le jeu de test | Final test evaluation |
| Le tour raté | Fuite (de données) | Data leakage |
| Le tour en premier, dans chaque monde | Chaîne de traitement | `Pipeline` |
| Choisir ses chapitres | Sélection de variables | Feature selection, `SelectKBest` |
| La fiche d'une page | Régularisation | Regularization |
| Raccourcir toutes les règles | Pénalité ℓ₂ | Ridge, `RidgeCV` |
| N'en garder que quelques-unes | Pénalité ℓ₁ | Lasso, `LassoCV`, `lasso_path` |
| La longueur de la page | Hyperparamètre λ | `alpha`, tuned by CV |
| Les chapitres photocopiés | Colinéarité, instabilité de la sélection | Multicollinearity |
| L'élève qui mémorise tout, sobrement | Interpolation à norme minimale, double descente | Min-norm interpolation, double descent |

---
*Fin de la journée. Le décor peut resservir tel quel pour réviser : rejouez les cellules, changez les graines, cherchez à mettre l'histoire en défaut.*